In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')
import sklearn.metrics as skm
import datetime as dt
import boto3
import pickle
import warnings
# Suppress PerformanceWarning
warnings.filterwarnings('ignore')

try:
    import catboost as cb
except:
    ! pip install catboost

In [2]:
print(f'Latest run date: {dt.datetime.now()}')

Latest run date: 2024-03-06 20:50:49.167623


### Functions

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

In [4]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_key, str_bucket_name):
    # upload file
    boto3.client('s3').upload_file(str_project, str_bucket_path, str_local_path)

In [5]:
# get tier
def get_tier(fltDebtorScore):
    if fltDebtorScore <= 0.04:
        return 'A1'
    elif fltDebtorScore <= 0.07:
        return 'A'
    elif fltDebtorScore <= 0.14:
        return 'B'
    elif fltDebtorScore <= 0.17:
        return 'C'
    elif fltDebtorScore <= 0.2015:
        return 'D'
    else:
        return 'Decline'

### Constants

In [6]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
str_dirname_output = './output'
str_variant = 'noPTImodel7'
flt_lgd_constant = 0.4324
flt_intercept_mike = -0.03281
flt_coef_mike = 1.95553

# indicators
dict_model_column = {
    'DQ1_1': 'Early_Pay_Delinquency_1_30_Flag',
    'DQ1_2': 'Early_Pay_Delinquency_1_60_Flag',
    'DQ1_3': 'Early_Pay_Delinquency_1_90_Flag',
    'DQ1_4': 'Early_Pay_Delinquency_1_120_Flag',
    'DQ1_5': 'Early_Pay_Delinquency_1_150_Flag',
    'DQ1_6': 'Early_Pay_Delinquency_1_180_Flag',
    'DQ1_7': 'Early_Pay_Delinquency_1_210_Flag',
    'DQ1_8': 'Early_Pay_Delinquency_1_240_Flag',
    'DQ1_9': 'Early_Pay_Delinquency_1_270_Flag',
    'DQ1_10': 'Early_Pay_Delinquency_1_300_Flag',
    'DQ1_11': 'Early_Pay_Delinquency_1_330_Flag',
    'DQ1_12': 'Early_Pay_Delinquency_1_360_Flag',
    'DQ1_13': 'Early_Pay_Delinquency_1_390_Flag',
    'DQ1_14': 'Early_Pay_Delinquency_1_420_Flag',
    'DQ1_15': 'Early_Pay_Delinquency_1_450_Flag',
    'DQ1_16': 'Early_Pay_Delinquency_1_480_Flag',
    'DQ1_17': 'Early_Pay_Delinquency_1_510_Flag',
    'DQ1_18': 'Early_Pay_Delinquency_1_540_Flag',
    'DQ1_19': 'Early_Pay_Delinquency_1_570_Flag',
    'DQ1_20': 'Early_Pay_Delinquency_1_600_Flag',
    'DQ1_21': 'Early_Pay_Delinquency_1_630_Flag',
    'DQ1_22': 'Early_Pay_Delinquency_1_660_Flag',
    'DQ1_23': 'Early_Pay_Delinquency_1_690_Flag',
    'DQ1_24': 'Early_Pay_Delinquency_1_720_Flag',
    'DQ15_1': 'Early_Pay_Delinquency_15_30_Flag',
    'DQ15_2': 'Early_Pay_Delinquency_15_60_Flag',
    'DQ15_3': 'Early_Pay_Delinquency_15_90_Flag',
    'DQ15_4': 'Early_Pay_Delinquency_15_120_Flag',
    'DQ15_5': 'Early_Pay_Delinquency_15_150_Flag',
    'DQ15_6': 'Early_Pay_Delinquency_15_180_Flag',
    'DQ15_7': 'Early_Pay_Delinquency_15_210_Flag',
    'DQ15_8': 'Early_Pay_Delinquency_15_240_Flag',
    'DQ15_9': 'Early_Pay_Delinquency_15_270_Flag',
    'DQ15_10': 'Early_Pay_Delinquency_15_300_Flag',
    'DQ15_11': 'Early_Pay_Delinquency_15_330_Flag',
    'DQ15_12': 'Early_Pay_Delinquency_15_360_Flag',
    'DQ15_13': 'Early_Pay_Delinquency_15_390_Flag',
    'DQ15_14': 'Early_Pay_Delinquency_15_420_Flag',
    'DQ15_15': 'Early_Pay_Delinquency_15_450_Flag',
    'DQ15_16': 'Early_Pay_Delinquency_15_480_Flag',
    'DQ15_17': 'Early_Pay_Delinquency_15_510_Flag',
    'DQ15_18': 'Early_Pay_Delinquency_15_540_Flag',
    'DQ15_19': 'Early_Pay_Delinquency_15_570_Flag',
    'DQ15_20': 'Early_Pay_Delinquency_15_600_Flag',
    'DQ15_21': 'Early_Pay_Delinquency_15_630_Flag',
    'DQ15_22': 'Early_Pay_Delinquency_15_660_Flag',
    'DQ15_23': 'Early_Pay_Delinquency_15_690_Flag',
    'DQ15_24': 'Early_Pay_Delinquency_15_720_Flag',
    'DQ30_1': 'Early_Pay_Delinquency_30_30_Flag',
    'DQ30_2': 'Early_Pay_Delinquency_30_60_Flag',
    'DQ30_3': 'Early_Pay_Delinquency_30_90_Flag',
    'DQ30_4': 'Early_Pay_Delinquency_30_120_Flag',
    'DQ30_5': 'Early_Pay_Delinquency_30_150_Flag',
    'DQ30_6': 'Early_Pay_Delinquency_30_180_Flag',
    'DQ30_7': 'Early_Pay_Delinquency_30_210_Flag',
    'DQ30_8': 'Early_Pay_Delinquency_30_240_Flag',
    'DQ30_9': 'Early_Pay_Delinquency_30_270_Flag',
    'DQ30_10': 'Early_Pay_Delinquency_30_300_Flag',
    'DQ30_11': 'Early_Pay_Delinquency_30_330_Flag',
    'DQ30_12': 'Early_Pay_Delinquency_30_360_Flag',
    'DQ30_13': 'Early_Pay_Delinquency_30_390_Flag',
    'DQ30_14': 'Early_Pay_Delinquency_30_420_Flag',
    'DQ30_15': 'Early_Pay_Delinquency_30_450_Flag',
    'DQ30_16': 'Early_Pay_Delinquency_30_480_Flag',
    'DQ30_17': 'Early_Pay_Delinquency_30_510_Flag',
    'DQ30_18': 'Early_Pay_Delinquency_30_540_Flag',
    'DQ30_19': 'Early_Pay_Delinquency_30_570_Flag',
    'DQ30_20': 'Early_Pay_Delinquency_30_600_Flag',
    'DQ30_21': 'Early_Pay_Delinquency_30_630_Flag',
    'DQ30_22': 'Early_Pay_Delinquency_30_660_Flag',
    'DQ30_23': 'Early_Pay_Delinquency_30_690_Flag',
    'DQ30_24': 'Early_Pay_Delinquency_30_720_Flag',
    'DQ60_3': 'Early_Pay_Delinquency_60_90_Flag',
    'DQ60_4': 'Early_Pay_Delinquency_60_120_Flag',
    'DQ60_5': 'Early_Pay_Delinquency_60_150_Flag',
    'DQ60_6': 'Early_Pay_Delinquency_60_180_Flag',
    'DQ60_7': 'Early_Pay_Delinquency_60_210_Flag',
    'DQ60_8': 'Early_Pay_Delinquency_60_240_Flag',
    'DQ60_9': 'Early_Pay_Delinquency_60_270_Flag',
    'DQ60_10': 'Early_Pay_Delinquency_60_300_Flag',
    'DQ60_11': 'Early_Pay_Delinquency_60_330_Flag',
    'DQ60_12': 'Early_Pay_Delinquency_60_360_Flag',
    'DQ60_13': 'Early_Pay_Delinquency_60_390_Flag',
    'DQ60_14': 'Early_Pay_Delinquency_60_420_Flag',
    'DQ60_15': 'Early_Pay_Delinquency_60_450_Flag',
    'DQ60_16': 'Early_Pay_Delinquency_60_480_Flag',
    'DQ60_17': 'Early_Pay_Delinquency_60_510_Flag',
    'DQ60_18': 'Early_Pay_Delinquency_60_540_Flag',
    'DQ60_19': 'Early_Pay_Delinquency_60_570_Flag',
    'DQ60_20': 'Early_Pay_Delinquency_60_600_Flag',
    'DQ60_21': 'Early_Pay_Delinquency_60_630_Flag',
    'DQ60_22': 'Early_Pay_Delinquency_60_660_Flag',
    'DQ60_23': 'Early_Pay_Delinquency_60_690_Flag',
    'DQ60_24': 'Early_Pay_Delinquency_60_720_Flag',
    'DQ90_4': 'Early_Pay_Delinquency_90_120_Flag',
    'DQ90_5': 'Early_Pay_Delinquency_90_150_Flag',
    'DQ90_6': 'Early_Pay_Delinquency_90_180_Flag',
    'DQ90_7': 'Early_Pay_Delinquency_90_210_Flag',
    'DQ90_8': 'Early_Pay_Delinquency_90_240_Flag',
    'DQ90_9': 'Early_Pay_Delinquency_90_270_Flag',
    'DQ90_10': 'Early_Pay_Delinquency_90_300_Flag',
    'DQ90_11': 'Early_Pay_Delinquency_90_330_Flag',
    'DQ90_12': 'Early_Pay_Delinquency_90_360_Flag',
    'DQ90_13': 'Early_Pay_Delinquency_90_390_Flag',
    'DQ90_14': 'Early_Pay_Delinquency_90_420_Flag',
    'DQ90_15': 'Early_Pay_Delinquency_90_450_Flag',
    'DQ90_16': 'Early_Pay_Delinquency_90_480_Flag',
    'DQ90_17': 'Early_Pay_Delinquency_90_510_Flag',
    'DQ90_18': 'Early_Pay_Delinquency_90_540_Flag',
    'DQ90_19': 'Early_Pay_Delinquency_90_570_Flag',
    'DQ90_20': 'Early_Pay_Delinquency_90_600_Flag',
    'DQ90_21': 'Early_Pay_Delinquency_90_630_Flag',
    'DQ90_22': 'Early_Pay_Delinquency_90_660_Flag',
    'DQ90_23': 'Early_Pay_Delinquency_90_690_Flag',
    'DQ90_24': 'Early_Pay_Delinquency_90_720_Flag',
}

# months (production data goes from 2021-09-27 to 2023-11-27)
list_str_year_month = [
    '2021-10',
    '2021-11',
    '2021-12',
    '2022-01',
    '2022-02',
#     '2022-03',
#     '2022-04',
#     '2022-05',
#     '2022-06',
#     '2022-07',
#     '2022-08',
#     '2022-09',
#     '2022-10',
#     '2022-11',
#     '2022-12',
#     '2023-01',
#     '2023-02',
#     '2023-03',
#     '2023-04',
#     '2023-05',
#     '2023-06',
#     '2023-07',
#     '2023-08',
#     '2023-09',
#     '2023-10',
]
# note: production targets were pulled 2024-03-06
# maximum days total of the targets is 720
# thus, to ensure all accounts have been 720 days (24 months) mature, I have subsetted to the latest date being 2022-02
# we can edit this as needed

Project: 20231010-gen-xii
Task: 09_early_indicators


### Output directory

In [7]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [8]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Get mean of target in training data

In [9]:
# load pd train data
str_filename = 'df_train_raw.gzip'
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
list_cols = [
    'uniqueid',
]
df_tmp = pd.read_parquet(str_uri, columns=list_cols)
df_tmp['uniqueid'] = df_tmp['uniqueid'].astype(int)
df_tmp.drop_duplicates(subset='uniqueid', keep='last', inplace=True)

# get targets
str_filename = 'df_monitoring_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
df_tmp2 = pd.read_csv(str_uri)
df_tmp2['UniqueID'] = df_tmp2['UniqueID'].astype(int)
df_tmp2.drop_duplicates(subset='UniqueID', keep='last', inplace=True)

# join
df_tmp = pd.merge(
    left=df_tmp,
    right=df_tmp2,
    left_on='uniqueid',
    right_on='UniqueID',
    how='inner'
)

# get means
dict_train_target_mean = {}
for str_col in tqdm(dict_model_column.keys()):
    # get mean
    flt_mean = df_tmp[str_col].mean()
    # assign
    dict_train_target_mean[str_col] = flt_mean

# save memory
del df_tmp

# show
dict_train_target_mean

100%|██████████| 115/115 [00:00<00:00, 7042.46it/s]


{'DQ1_1': 0.19842143595128808,
 'DQ1_2': 0.36320523026371504,
 'DQ1_3': 0.4827154011619044,
 'DQ1_4': 0.5659965108992057,
 'DQ1_5': 0.6263274673531952,
 'DQ1_6': 0.6715164038549483,
 'DQ1_7': 0.7060009146186549,
 'DQ1_8': 0.731830422926441,
 'DQ1_9': 0.7541030809098762,
 'DQ1_10': 0.7707355905218407,
 'DQ1_11': 0.784810555376772,
 'DQ1_12': 0.7966667231246083,
 'DQ1_13': 0.8074388983926424,
 'DQ1_14': 0.8173811419183279,
 'DQ1_15': 0.8252908995443844,
 'DQ1_16': 0.8317609796582036,
 'DQ1_17': 0.8379431242695754,
 'DQ1_18': 0.8429565894886604,
 'DQ1_19': 0.8479869920902424,
 'DQ1_20': 0.8525262105994139,
 'DQ1_21': 0.8563201842787216,
 'DQ1_22': 0.8596399112481157,
 'DQ1_23': 0.8630104503650006,
 'DQ1_24': 0.8659914296844565,
 'DQ15_1': 0.010399552853102081,
 'DQ15_2': 0.038159922765535814,
 'DQ15_3': 0.07083213360207313,
 'DQ15_4': 0.10457139953591572,
 'DQ15_5': 0.1392422215070883,
 'DQ15_6': 0.1731847360308938,
 'DQ15_7': 0.20350265070036078,
 'DQ15_8': 0.23007740383801087,
 'DQ15_9'

### Load data from retro scoring

In [10]:
str_filename = 'df.gzip'
str_uri = f's3://{str_project}/08_retro_scoring/06_create_df/{str_filename}'
df = pd.read_parquet(str_uri)

# convert to datetime
df['applicationdate__app'] = pd.to_datetime(df['applicationdate__app'])

# drop because they break preprocessing
list_cols = [
    'applicationdayofweek__app',
]
df.drop(list_cols, axis=1, inplace=True)

# get features 100% NaN and drop
ser_isnull = df.isnull().mean()
list_all_nan = list(ser_isnull[ser_isnull==1.0].index)
# logic
if str_variant == 'model3':
    list_all_nan = [col for col in list_all_nan if col != 'cvlst_s1__tu']
else:
    pass
df.drop(list_all_nan, axis=1, inplace=True)

# get month of application
df['year_month'] = df['applicationdate__app'].dt.strftime('%Y-%m')
# subset
df = df[df['year_month'].isin(list_str_year_month)].copy()

# show
df

,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,year_month
0,5838462__7328771__20211124,Burlington,Kentucky,41005,True,nan,True,11,4,2836.0,...,500.0,484.52,0.348085,0.124236,0,0.955681,auto,0,2013-05-08 08:47:59.997,2021-11
1,5874043__7369806__20211220,Dallas,Texas,75219,True,nan,True,12,4,1469.0,...,150.0,599.41,0.413636,0.137929,0,1.118242,suv,1,2010-07-07 16:51:47.837,2021-12
2,5827507__7315370__20211113,SAINT LOUIS,Missouri,63112,True,nan,True,11,4,5473.0,...,1000.0,438.12,0.320558,0.110788,0,1.092707,auto,1,2018-05-07 13:27:18.220,2021-11
3,5855839__7349888__20211211,COLLINSVILLE,Illinois,62234,True,nan,True,12,4,5235.0,...,2000.0,386.01,0.364331,0.122209,0,1.097128,auto,1,2017-09-18 16:02:07.110,2021-12
4,5874497__7370346__20211220,CHICAGO,Illinois,60652,True,nan,True,12,4,4535.0,...,1500.0,680.00,0.160800,0.068000,0,0.957552,suv,0,2016-03-15 15:36:18.820,2021-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9417,0__0__20220222,CHANDLER,Arizona,85225,False,0.0,False,2,1,NaN,...,1000.0,510.48,0.349799,0.149798,0,1.122614,suv,0,2003-07-08 14:32:29.097,2022-02
9525,0__0__20220221,Indianapolis,Indiana,46237,False,0.0,False,2,1,NaN,...,1500.0,424.84,0.474754,0.073374,0,0.917676,suv,0,2013-07-18 09:37:34.627,2022-02
9702,0__0__20220222,Louisville,Kentucky,40272,False,0.0,False,2,1,NaN,...,0.0,529.30,0.436436,0.112239,0,0.923187,auto,1,2012-11-21 13:38:42.487,2022-02
10670,0__0__20220214,CYPRESS,Louisiana,71457,False,0.0,False,2,1,NaN,...,0.0,575.00,0.403836,0.123374,0,1.097798,auto,0,2012-06-18 09:31:54.393,2022-02


### Date range

In [11]:
dtm_min = df['applicationdate__app'].min()
dtm_max = df['applicationdate__app'].max()
print(f'Min date: {dtm_min}')
print(f'Max date: {dtm_max}')

Min date: 2021-10-18 13:15:40.490000
Max date: 2022-02-28 21:43:21.703000


### Preprocess

In [12]:
# # preprocess
# list_str_filename = [
#     'preprocessing.py',
#     'cls_model_preprocessing.pkl',
# ]
# for str_filename in tqdm(list_str_filename):
#     # download
#     str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
#     str_local_path = f'./{str_filename}'
#     download_from_s3(
#         str_local_path=str_local_path, 
#         str_bucket_path=str_bucket_path, 
#         str_project=str_project,
#     )
# # import
# cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
# # rm
# os.remove(str_local_path)

# # preprocess
# df = cls_model_preprocessing.transform(df)

# # rm
# os.remove('./preprocessing.py')

# # show
# df

### Predict

In [13]:
# # get PD model
# str_filename = 'final_model.pkl'
# str_bucket_path = f'02_pricing_pd/02_model/{str_variant}/03_final_model/{str_filename}'
# str_local_path = f'{str_dirname_output}/{str_filename}'
# download_from_s3(
#     str_local_path=str_local_path, 
#     str_bucket_path=str_bucket_path, 
#     str_project=str_project,
# )
# cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
# os.remove(str_local_path)
# # predict
# list_cols_model = list(cls_model_inference.feature_names_)
# df['yhat'] = cls_model_inference.predict_proba(df[list_cols_model])[:, 1]
# # show
# df

### Get tier

In [14]:
# # get ecnl
# df['ecnl'] = df['yhat'] * flt_lgd_constant
# # get modified ecnl
# df['ecnl_mod'] = flt_intercept_mike + (flt_coef_mike * df['ecnl'])
# # get tier
# df['tier'] = df['ecnl_mod'].apply(get_tier)
# # subset
# df = df[df['tier'] != 'Decline'].copy()
# # show
# df

### Get actual targets pulled from the DB today

In [15]:
# get actual targets
str_filename = 'df_early_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/05_get_target_from_db/{str_filename}'
df_tmp = pd.read_csv(str_uri)

# join
df = pd.merge(
    left=df,
    right=df_tmp,
    on='bigAccountId',
    how='inner',
)

dict_prod_target_mean = {}
for key, val in tqdm(dict_model_column.items()):
    dict_prod_target_mean[key] = df[val].mean()

# show
df

100%|██████████| 115/115 [00:00<00:00, 15820.81it/s]


,uniqueid__app_x,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,bigdealerid__app,...,Early_Pay_Delinquency_90_450_Flag,Early_Pay_Delinquency_90_480_Flag,Early_Pay_Delinquency_90_510_Flag,Early_Pay_Delinquency_90_540_Flag,Early_Pay_Delinquency_90_570_Flag,Early_Pay_Delinquency_90_600_Flag,Early_Pay_Delinquency_90_630_Flag,Early_Pay_Delinquency_90_660_Flag,Early_Pay_Delinquency_90_690_Flag,Early_Pay_Delinquency_90_720_Flag
0,5838462__7328771__20211124,Burlington,Kentucky,41005,True,nan,True,11,4,2836.0,...,1,1,1,1,1,1,1,1,1,1
1,5874043__7369806__20211220,Dallas,Texas,75219,True,nan,True,12,4,1469.0,...,0,1,1,1,1,1,1,1,1,1
2,5827507__7315370__20211113,SAINT LOUIS,Missouri,63112,True,nan,True,11,4,5473.0,...,1,1,1,1,1,1,1,1,1,1
3,5855839__7349888__20211211,COLLINSVILLE,Illinois,62234,True,nan,True,12,4,5235.0,...,0,0,0,1,1,1,1,1,1,1
4,5874497__7370346__20211220,CHICAGO,Illinois,60652,True,nan,True,12,4,4535.0,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4891,0__0__20220222,CHANDLER,Arizona,85225,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0
4892,0__0__20220221,Indianapolis,Indiana,46237,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0
4893,0__0__20220222,Louisville,Kentucky,40272,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0
4894,0__0__20220214,CYPRESS,Louisiana,71457,False,0.0,False,2,1,NaN,...,0,0,0,0,0,0,0,0,0,0


### Summary

In [16]:
list_dict_row = []
for key, val in tqdm(dict_model_column.items()):
    # get mean training target
    flt_mean_train_target = dict_train_target_mean[key]
    # get mean of production actuals
    flt_mean_prod_actual = dict_prod_target_mean[key]
    # dict_row
    dict_row = {
        'indicator': key,
        'mean_train_target': flt_mean_train_target,
        'mean_prod_actual': flt_mean_prod_actual,
    }
    list_dict_row.append(dict_row)
    
# make df 
df_summary = pd.DataFrame(list_dict_row)

# dates
df_summary['min_date'] = list_str_year_month[0]
df_summary['max_date'] = list_str_year_month[-1]

# get days delinquent
df_summary['days_delinquent'] = df_summary['indicator'].apply(
    lambda x: int(x.split('_')[0][2:]),
)

# get days total
df_summary['days_total'] = df_summary['indicator'].apply(
    lambda x: int(x.split('_')[1]) * 30, # 30 days per month
)

# show
df_summary

100%|██████████| 115/115 [00:00<00:00, 663473.12it/s]


,indicator,mean_train_target,mean_prod_actual,min_date,max_date,days_delinquent,days_total
0,DQ1_1,0.198421,0.004085,2021-10,2022-02,1,30
1,DQ1_2,0.363205,0.443627,2021-10,2022-02,1,60
2,DQ1_3,0.482715,0.610498,2021-10,2022-02,1,90
3,DQ1_4,0.565997,0.719975,2021-10,2022-02,1,120
4,DQ1_5,0.626327,0.783905,2021-10,2022-02,1,150
...,...,...,...,...,...,...,...
110,DQ90_20,0.111380,0.346609,2021-10,2022-02,90,600
111,DQ90_21,0.120001,0.362337,2021-10,2022-02,90,630
112,DQ90_22,0.128267,0.378676,2021-10,2022-02,90,660
113,DQ90_23,0.137311,0.384600,2021-10,2022-02,90,690


### Save

In [17]:
str_filename = 'df_summary.csv'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
df_summary.to_csv(str_local_path, index=False)